# hcw_lasso

> 对应代码：`EconometricsCode/code_in_notes/Chap.15/hcw_lasso.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们首先清理环境并读入分省份的年度出口面板数据 shannxi_export.dta，随后用 tsset 将数据声明为以 year 为指标的时间序列，这样后面才能使用 tsline 等时间序列命令。

In [ ]:
%%stata
clear
set more off
use ../datasets/shannxi_export.dta
tsset year

order 命令把 year 与处理组变量 export_locationShaanxi（陕西省出口）排到数据最前面，方便查看；真正的变量变换在后面的循环中完成。

In [ ]:
%%stata
order year export_locationShaanxi

这个 foreach 循环对从陕西到浙江的所有省份出口变量统一做两步处理：先取对数，把出口额转化为对数形式以缓解规模差异并便于解释为增长率；再用 egen 的 std() 函数把每个序列标准化为均值 0、方差 1，生成以 std_ 为前缀的新变量。标准化使得各省份序列在 Lasso 中具有可比的量纲，避免因量纲不同而导致惩罚不公平。注意循环体用花括号包裹，不可拆分。

In [ ]:
%%stata
foreach v of varlist export_locationShaanxi- export_locationZhejiang{
	replace `v'=log(`v')
	egen std_`v'=std(`v')
}

我们把候选对照组（其他省份的标准化出口序列）存入局部宏 control。这些省份将作为 HCW 方法中构造陕西省反事实出口路径的“控制变量池”。

In [ ]:
%%stata
local control "std_export_locationShanghai std_export_locationShanxi std_export_locationFujian std_export_locationGuangdong std_export_locationGuizhou std_export_locationHeilongjiang std_export_locationJiangxi std_export_locationAnhui"

这是 HCW（Hsiao, Ching, Wan, 2012）方法的核心第一步：只用处理前（2013 年中欧班列开通之前）的数据，以陕西省出口为被解释变量、其他省份出口为候选解释变量做线性回归，并用 Lasso 的 L1 惩罚在多个候选对照中自动选出与陕西出口路径最相关的少数省份，以避免过拟合。if year<2013 限定只用政策发生前的样本进行变量选择。

In [ ]:
%%stata
lasso linear export_locationShaanxi `control' if year<2013

lasso 是后估计命令丰富的命令，我们通过 e(post_sel_vars) 取出被 Lasso 选中的变量名单，存入局部宏 selected_vars 备用。

In [ ]:
%%stata
local selected_vars=e(post_sel_vars)

predict 利用刚才估计的 Lasso 模型对全样本（包括政策发生之后）生成预测值 p_shaanxi_lasso。这一预测序列就是 Lasso 版 HCW 方法给出的陕西省出口的“反事实”路径——即假设中欧班列没有开通时陕西出口本该走的轨迹，政策效应即为实际值与该反事实之差。

In [ ]:
%%stata
predict p_shaanxi_lasso

作为对照，我们再实现 Li 和 Bell（2017）建议之前的原始 HCW 做法：用 Lasso 选出的那组省份做普通最小二乘回归（同样只用 2013 年前的数据估计系数），然后对全样本预测，得到 OLS 版的反事实路径 p_shaanxi。两条反事实路径画在一起可以直观比较 Lasso 直接预测与“Lasso 选变量 + OLS 估计”两种策略的差异。

In [ ]:
%%stata
reg `selected_vars' if year<2013
predict p_shaanxi

我们为三条序列加上中文标签，便于图例显示：实际出口、OLS 反事实、Lasso 反事实。

In [ ]:
%%stata
label variable export_locationShaanxi "陕西省实际出口"
label variable p_shaanxi "陕西省出口的反事实估计"
label variable p_shaanxi_lasso "陕西省出口的反事实估计Lasso"

最后用 tsline 把陕西省实际出口与两条反事实路径画在同一张时间序列图上：lp(solid dash) 指定实线与虚线区分，xline(2013 2016) 在 2013 年（中欧班列开通）与 2016 年处画出竖直参考线以标示政策时点，scheme(s1mono) 使用适合打印的黑白模板。政策发生后实际值与反事实值之间的垂直距离即为中欧班列对陕西出口的（HCW 意义下的）处理效应。

In [ ]:
%%stata
tsline export_locationShaanxi p_shaanxi p_shaanxi_lasso, lp(solid dash) xline(2013 2016, lstyle(grid)) scheme(s1mono) xtitle("年份") ytitle("出口对数值")